In [ ]:
import json
from datetime import datetime
from matplotlib import pyplot as plt
import pycountry

from emu_renewal.inputs import DATA_PATH, get_hosp_series_from_owid_data

In [ ]:
included = json.load(open(DATA_PATH / "config/included.json", "r"))
start_time = datetime(2020, 4, 1)
end_time = datetime(2021, 6, 30)

In [ ]:
def get_country_hosps(country, start, end):
    admits = get_hosp_series_from_owid_data("Weekly new hospital admissions", country)
    filt_admits = admits[(start < admits.index) & (admits.index < end)]
    occup = get_hosp_series_from_owid_data("Daily hospital occupancy", country)
    filt_occup = occup[(start < occup.index) & (occup.index < end)]
    icu_admits = get_hosp_series_from_owid_data("Weekly new ICU admissions", country)
    filt_icu_admits = icu_admits[(start < icu_admits.index) & (icu_admits.index < end)]
    icu_occup = get_hosp_series_from_owid_data("Daily ICU occupancy", country)
    filt_icu_occup = icu_occup[(start < icu_occup.index) & (icu_occup.index < end)]
    if not filt_admits.empty:
        return filt_admits, "admits"
    elif not filt_occup.empty:
        return filt_occup, "occup"
    elif not filt_icu_admits.empty:
        return filt_icu_admits, "icu_admits"
    elif not filt_icu_occup.empty:
        return filt_icu_occup, "icu_occup"
    else:
        return None, ""

In [ ]:
hosp_data["DZA"][::7]

In [ ]:
hosp_data = {}
ind = {}
for c in included:
    data, indicator = get_country_hosps(c, start_time, end_time)
    hosp_data[c] = data
    ind[c] = indicator

In [ ]:
fig, axes = plt.subplots(10, 9, figsize=(20, 20))
flat_axes = axes.ravel()
for i, (c, data) in enumerate(hosp_data.items()):
    ax = flat_axes[i]
    title = pycountry.countries.lookup(c).name
    if data is not None:
        ax.plot(data.index, data)
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=70)
        title = title + f", {ind[c]}"
    ax.set_title(title)
fig.tight_layout()